<a href="https://colab.research.google.com/github/444112029012/phishing-detection-project/blob/main/colab/%E5%89%B5%E5%BB%BA%E8%B3%87%E6%96%99%E9%9B%86/HTML_play_wright.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
!pip install playwright
!playwright install chromium
!playwright install-deps

In [ ]:
import pandas as pd
import aiohttp
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
import re
import asyncio
import os
import numpy as np
import gc
import ipaddress
from typing import Tuple, Optional
from google.colab import drive

# ====== Playwright Async 模組 ======
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError, Error as PlaywrightError

# --- Google Drive 掛載 ---
drive.mount('/content/drive')

FILE_NAME = "/content/drive/MyDrive/畢業專題Playwright_html/phishing_dataset_expansion_forEmbeddingModule_html.csv"

# --- ⚙️ 第二招：併發與批次設定 ---
CONCURRENCY_LIMIT = 50   # 同時開啟的分頁數 (Colab 免費版建議 5~10)
BATCH_SIZE = 200        # 每處理幾筆存一次檔
RESTART_INTERVAL = 600  # 每處理幾筆強制重啟瀏覽器釋放記憶體
RENDER_WAIT_TIME = 2

# 建立號誌，限制同時執行的任務數量
semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

async def block_agressive_resources(route):
    """阻擋圖片、影片、CSS 載入，大幅提升速度"""
    if route.request.resource_type in ["image", "media", "font", "stylesheet"]:
        await route.abort()
    else:
        await route.continue_()

async def get_html_content_async(session, url, timeout=20, max_retries=2):
    """使用 aiohttp 快速獲取靜態網頁"""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.7258.66 Safari/537.36'}
    for attempt in range(max_retries):
        try:
            async with session.get(url, headers=headers, timeout=timeout, ssl=False) as response:
                if response.status >= 400:
                    return None, False

                html_content = await response.text()
                # 第三招：改用 lxml 加速解析
                soup = BeautifulSoup(html_content, 'lxml')
                body_content = soup.find('body')

                if html_content and len(html_content.strip()) > 100:
                    not_found_keywords = ['page not found', 'error 404', 'page does not exist', '找不到頁面', '頁面不存在']
                    text_to_check = body_content.get_text(strip=True).lower() if body_content else html_content.lower()
                    matched_keywords = sum(1 for kw in not_found_keywords if kw in text_to_check)

                    if matched_keywords >= 2:
                        return None, False
                    return html_content, False
                else:
                    break
        except Exception:
            if attempt == max_retries - 1: break
            await asyncio.sleep(1)
    return None, None

async def fetch_dynamic_content_async(page, url: str) -> Tuple[Optional[str], str]:
    """使用 Playwright 獲取動態網頁 (包含轉址處理)"""
    try:
        try:
            response = await page.goto(url, timeout=30000, wait_until='domcontentloaded')
        except PlaywrightError as e:
            if "interrupted by another navigation" in str(e):
                try: await page.wait_for_load_state('domcontentloaded', timeout=20000)
                except: pass
            else:
                raise e

        await page.wait_for_timeout(RENDER_WAIT_TIME * 1000)

        # 簡單滾動一下
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight / 2);")
        await page.wait_for_timeout(50)

        text = await page.content()
        return (text, 'OK_Dynamic') if text else (None, 'OK_Dynamic_Empty')

    except Exception:
        return None, 'Error_Playwright'

async def process_single_row(index, row, session, context, df, html_feature_columns, total_rows):
  """處理單筆資料的工作節點 (Worker)"""
  url = row['url']
  if pd.isna(url) or not str(url).strip():
    df.at[index, 'feature_extracted'] = 0.0
    return

  if not url.startswith('http://') and not url.startswith('https://'):
    url = 'http://' + url

  # 每個任務獨立開一個 Page，避免互相干擾
  page = await context.new_page()
  await page.route("**/*", block_agressive_resources)

  try:
    print(f"🔍 處理中 [{index+1}/{total_rows}]: {url}")

    html_content, _ = await get_html_content_async(session, url, timeout=15, max_retries=1)
    dynamic_content = None
    if html_content:
      dynamic_content, _ = await fetch_dynamic_content_async(page, url)

    if html_content and dynamic_content:
      # ====== 以下是你的特徵解析邏輯 ======
      # 第三招：統一使用 'lxml' 加速
      soup_static = BeautifulSoup(html_content, 'lxml')
      soup_dynamic = BeautifulSoup(dynamic_content, 'lxml')

      parsed_url = urlparse(url)
      base_domain = parsed_url.netloc.split(':')[0]
      if base_domain.startswith('www.'): base_domain = base_domain[4:]

      # 15 & 16
      meta_tag = soup_static.find('meta', attrs={'http-equiv': lambda x: x and x.lower() == 'refresh'})
      df.at[index, 'has_meta_refresh'] = 1.0 if meta_tag and "url=" in meta_tag.get("content", "").lower() else 0.0
      redirect_kws = ["window.location.href", "location.href", "location.assign", "location.replace"]
      df.at[index, 'has_js_redirect'] = 1.0 if soup_static.find("script", string=lambda s: any(k in s for k in redirect_kws) if s else False) else 0.0

      # 1
      phish_kws = ['login', 'signin', 'account update', 'verify account', 'security alert', 'password', 'bank', 'paypal']
      text_content = soup_dynamic.get_text().lower()
      df.at[index, 'phish_hints'] = 1.0 if any(kw in text_content for kw in phish_kws) else 0.0

      # 2
      domain_parts = base_domain.split('.')
      core_domain = domain_parts[-2] if len(domain_parts) >= 2 and domain_parts[-1] in ['com', 'org', 'net', 'edu', 'gov'] else (domain_parts[-3] if len(domain_parts) >= 3 else domain_parts[0])
      title_tag = soup_dynamic.find('title')
      meta_desc = soup_dynamic.find('meta', attrs={'name': 'description'})
      df.at[index, 'domain_in_brand'] = 1.0 if (title_tag and core_domain in title_tag.get_text().lower()) or (meta_desc and core_domain in meta_desc.get('content', '').lower()) else 0.0

      # 3, 4, 5, 6, 7, 11
      all_links = soup_dynamic.find_all('a', href=True)
      df.at[index, 'nb_hyperlinks'] = len(all_links)

      int_links, ext_links, red_count, err_count = 0, 0, 0, 0
      is_safe_anchor = 1.0
      suspicious_kws = ['bit.ly', 'tinyurl', 'goo.gl', 't.co']

      for link in all_links:
          href = link['href']
          if href.startswith('#'): continue
          full_url = urljoin(url, href)
          linked_domain = urlparse(full_url).netloc
          if linked_domain == parsed_url.netloc:
            int_links += 1
          else:
            ext_links += 1
            if link.get('onclick') and 'window.location' in link.get('onclick', ''): red_count += 1
            elif link.get('target') == '_blank' and 'redirect' in link.get_text().lower(): red_count += 1
            if 'error' in full_url.lower() or '404' in full_url or not linked_domain: err_count += 1

            try: ipaddress.ip_address(linked_domain); is_safe_anchor = 0.0
            except ValueError: pass
            if any(kw in linked_domain.lower() for kw in suspicious_kws): is_safe_anchor = 0.0

      total_links = int_links + ext_links
      df.at[index, 'ratio_intHyperlinks'] = int_links / total_links if total_links > 0 else 0.0
      df.at[index, 'ratio_extHyperlinks'] = ext_links / total_links if total_links > 0 else 0.0
      df.at[index, 'ratio_extRedirection'] = red_count / len(all_links) if all_links else 0.0
      df.at[index, 'ratio_extErrors'] = err_count / len(all_links) if all_links else 0.0
      df.at[index, 'safe_anchor'] = is_safe_anchor

      # 8, 9, 10
      favicon = soup_dynamic.find('link', rel=lambda x: x and 'icon' in x.lower())
      df.at[index, 'external_favicon'] = 1.0 if favicon and 'href' in favicon.attrs and urlparse(urljoin(url, favicon['href'])).netloc != parsed_url.netloc else 0.0
      df.at[index, 'links_in_tags'] = sum(1 for tag in soup_dynamic.find_all(['a', 'script', 'img', 'link', 'iframe', 'form']) if 'href' in tag.attrs or 'src' in tag.attrs or (tag.name == 'form' and 'action' in tag.attrs))

      media_tags = soup_dynamic.find_all(['img', 'audio', 'video', 'source'])
      ext_media = sum(1 for tag in media_tags if (tag.get('src') or tag.get('href')) and urlparse(urljoin(url, tag.get('src') or tag.get('href'))).netloc != parsed_url.netloc)
      df.at[index, 'ratio_extMedia'] = ext_media / len(media_tags) if media_tags else 0.0

      # 12, 13, 14
      df.at[index, 'empty_title'] = 1.0 if not (title_tag and title_tag.string and title_tag.string.strip()) else 0.0
      df.at[index, 'domain_in_title'] = 1.0 if title_tag and title_tag.string and base_domain in title_tag.string.lower() else 0.0
      copyright_tags = soup_dynamic.find_all(text=re.compile(r'©|copyright|all rights reserved', re.IGNORECASE))
      df.at[index, 'domain_with_copyright'] = 1.0 if any(base_domain in tag.lower() for tag in copyright_tags) else 0.0

      df.at[index, 'feature_extracted'] = 1.0

    else:
      for col in html_feature_columns: df.at[index, col] = 0.0
      df.at[index, 'feature_extracted'] = 0.0

  except Exception as e:
    for col in html_feature_columns: df.at[index, col] = 0.0
    df.at[index, 'feature_extracted'] = 0.0
  finally:
    # ✅ 確保用完的 Page 一定會被關閉，釋放記憶體！
    await page.close()

async def safe_worker_wrapper(index, row, session, context, df, html_feature_columns, total_rows):
    # 先拿到「並發許可證（入座）」，才開始算 90 秒！
    async with semaphore:
        try:
            # 入座後，給他 90 秒的時間吃拉麵（爬蟲）
            await asyncio.wait_for(
                process_single_row(index, row, session, context, df, html_feature_columns, total_rows),
                timeout=90.0
            )
        except asyncio.TimeoutError:
            print(f"⏰ [上帝大限] 第 {index+1} 筆 URL ({row['url']}) 卡死超過 90 秒，已強制拔管！")
            df.at[index, 'feature_extracted'] = 0.0
            for col in html_feature_columns:
                if col != 'feature_extracted':
                    df.at[index, col] = 0.0
        except Exception as e:
            print(f"❌ [未預期崩潰] 第 {index+1} 筆 URL 發生錯誤: {e}")
            df.at[index, 'feature_extracted'] = 0.0

async def process_dataset(df: pd.DataFrame) -> pd.DataFrame:
    html_feature_columns = [
        'phish_hints', 'domain_in_brand', 'nb_hyperlinks', 'ratio_intHyperlinks',
        'ratio_extHyperlinks', 'ratio_extRedirection', 'ratio_extErrors',
        'external_favicon', 'links_in_tags', 'ratio_extMedia', 'safe_anchor',
        'empty_title', 'domain_in_title', 'domain_with_copyright',
        'has_meta_refresh', 'has_js_redirect', 'feature_extracted'
    ]

    if 'feature_extracted' not in df.columns:
        df[html_feature_columns] = np.nan

    total_rows = len(df)
    print(f"總共 {total_rows} 筆資料準備進入併發處理...")

    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(ssl=False)) as session:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True, args=['--no-sandbox', '--disable-dev-shm-usage'])
            context = await browser.new_context(ignore_https_errors=True)

            try:
                # 🔪 將任務切成 BATCH_SIZE 大小的批次
                for i in range(0, total_rows, BATCH_SIZE):
                    batch_df = df.iloc[i:i+BATCH_SIZE]
                    tasks = []

                    for index, row in batch_df.iterrows():
                        # 跳過已經成功提取特徵的資料 (斷點續傳)
                        # if index < 2400:
                        #   continue
                        if row.get('feature_extracted') == 1.0:
                            continue

                        # 建立併發任務
                        task = asyncio.create_task(
                            safe_worker_wrapper(index, row, session, context, df, html_feature_columns, total_rows)
                        )
                        tasks.append(task)

                    # 如果這個批次有任務需要跑，就一口氣執行它們
                    if tasks:
                        await asyncio.gather(*tasks, return_exceptions=True)

                    # 批次結束，進行存檔
                    print(f"\n💾 --- 已處理至 {min(i+BATCH_SIZE, total_rows)} 筆，寫入 Google Drive 中... ---")
                    df.to_csv(FILE_NAME, index=False)

                    # 記憶體回收機制
                    if i > 0 and i % RESTART_INTERVAL == 0:
                        print("♻️ 達到重啟門檻，重建瀏覽器 Context 釋放深層記憶體...")
                        await context.close()
                        gc.collect()
                        context = await browser.new_context(ignore_https_errors=True)

            except (KeyboardInterrupt, asyncio.CancelledError):
                print("\n🛑 偵測到手動中斷，儲存最終進度...")
                df.to_csv(FILE_NAME, index=False)
            finally:
                await browser.close()
    return df

async def main():
    os.system("pkill -9 -f chrome")

    if os.path.exists(FILE_NAME):
        print(f"✅ 找到 Drive 中的進度檔: {FILE_NAME}")
        df = pd.read_csv(FILE_NAME)
        # 如果你想「全部重來」，把下面這行取消註解：
        # df['feature_extracted'] = np.nan
    else:
        print(f"⚠️ 找不到進度檔。請確保原始 CSV 存在，或者修改讀取路徑。")
        # 這裡請替換成你的「最原始資料」的讀取路徑
        # df = pd.read_csv("/content/原本的原始檔.csv")
        return

    df_updated = await process_dataset(df)
    print("🎉 任務徹底完成！")

# 啟動任務
await main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 找到 Drive 中的進度檔: /content/drive/MyDrive/畢業專題Playwright_html/phishing_dataset_expansion_forEmbeddingModule_html.csv
總共 100000 筆資料準備進入併發處理...


ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


🔍 處理中 [4/100000]: https://www.sfnmjournal.com
🔍 處理中 [15/100000]: https://www.bulgariaski.com
🔍 處理中 [10/100000]: https://www.aap.org
🔍 處理中 [17/100000]: https://www.motley.ie
🔍 處理中 [22/100000]: http://www.shprakserf.gq
🔍 處理中 [21/100000]: http://www.f0519141.xsph.ru
🔍 處理中 [26/100000]: https://www.bwresearch.com
🔍 處理中 [28/100000]: https://service-mitld.firebaseapp.com/
🔍 處理中 [29/100000]: http://www.kuradox92.lima-city.de
🔍 處理中 [30/100000]: https://liuy-9a930.web.app/
🔍 處理中 [33/100000]: http://att-103731-107123.weeblysite.com/
🔍 處理中 [32/100000]: https://ipfs.io/ipfs/qmrvvyr84esa2assw9vvwupqjgsdn4c3dwkusfdwzdz3kn?clientid=noc@protocol.ai
🔍 處理中 [35/100000]: https://hidok4f8zl.firebaseapp.com/
🔍 處理中 [41/100000]: http://www.iuhjn.pplink.club
🔍 處理中 [42/100000]: https://mechinchem-5cb8a.web.app/
🔍 處理中 [45/100000]: https://fb-restriction-case-97be5.web.app/
🔍 處理中 [46/100000]: https://pontosapontamentolu.com/gclid/=/c/?gclid=ishecabh95cvbhb1eiwagy6m4vn7url3tlj5m_nu__lcyj4n06pqquydbh-56148buwwbmv7k1

/tmp/ipykernel_5875/4000765853.py:49: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_content, 'lxml')



💾 --- 已處理至 1000 筆，寫入 Google Drive 中... ---
🔍 處理中 [1008/100000]: http://www.webmail.yourturbe.org
🔍 處理中 [1005/100000]: http://www.coinbasewalletones.com
🔍 處理中 [1011/100000]: https://www.rvtravel.com
🔍 處理中 [1013/100000]: http://www.pmdu.info
🔍 處理中 [1012/100000]: https://www.amazcon-co-jp.amacszan.niixwp.top/ap/signin
🔍 處理中 [1016/100000]: https://www.tdrtargets.org
🔍 處理中 [1021/100000]: https://www.hanbit.co.kr
🔍 處理中 [1028/100000]: http://dj-wedding.tk/besst/
🔍 處理中 [1029/100000]: https://sweetflippantaddin.rgystre3.repl.co/des/index.php
🔍 處理中 [1032/100000]: http://www.vhidsods.ga
🔍 處理中 [1026/100000]: https://www.loveokko.co
🔍 處理中 [1024/100000]: https://dev-bhdleonconfiguracionrd.pantheonsite.io/
🔍 處理中 [1019/100000]: http://www.wispy-fire-1da3.nscimupf.workers.dev
🔍 處理中 [1014/100000]: https://www.1000-ideen.at
🔍 處理中 [1030/100000]: https://www.postaprodeti.cz
🔍 處理中 [1033/100000]: https://mail-101591.weeblysite.com/
🔍 處理中 [1034/100000]: http://paxful-wap.vip
🔍 處理中 [1039/100000]: https://webm

...

🔍 處理中 [8781/100000]: https://www.zzyzx.co.jp
🔍 處理中 [8782/100000]: https://www.finehomedisplays.com
🔍 處理中 [8783/100000]: https://attjhdgdj-102860.square.site/
🔍 處理中 [8784/100000]: http://www.d0054262.atservers.net/
🔍 處理中 [8785/100000]: https://www.bonvion.com
⏰ [上帝大限] 第 8696 筆 URL (http://www.cyberinc.nl) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 8697 筆 URL (https://www.royaldutchmint.com) 卡死超過 90 秒，已強制拔管！
🔍 處理中 [8786/100000]: https://www.cinnox.com
⏰ [上帝大限] 第 8722 筆 URL (https://www.bestanimations.com) 卡死超過 90 秒，已強制拔管！
🔍 處理中 [8787/100000]: https://vinted-pl-gj32d.bioxyn.top/processing/1673439760393740
🔍 處理中 [8788/100000]: https://www.cforcode.com
🔍 處理中 [8789/100000]: http://43.128.92.128/interactivelogin?continue=https://accounts.google.com/?&amp;xrealip=35.203.252.150&amp;ifkv=awnogheuobkba6c5lfvrocifv2lbpyjrrcmw4ukzlek14meqbpve-oyqca1rvdvg6c_573zaovjflg
🔍 處理中 [8790/100000]: https://www.tees.ac.uk
🔍 處理中 [8791/100000]: http://www.gutmannr.ga
🔍 處理中 [8792/100000]: http://www.reviewer.mobi
🔍 處理中 [8793/

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed


🔍 處理中 [8945/100000]: https://www.anglicantheologicalreview.org
🔍 處理中 [8946/100000]: https://sbi-1100ab.firebaseapp.com/
🔍 處理中 [8948/100000]: https://www.fracturedatlas.org
🔍 處理中 [8947/100000]: https://www.apwu.org
🔍 處理中 [8949/100000]: https://www.coffeegallery.com
🔍 處理中 [8950/100000]: https://manages-hostinges.studiorevelli.it/pec/managehosting/
🔍 處理中 [8951/100000]: http://www.pay-pamant.ga
🔍 處理中 [8952/100000]: https://30-02-5g90-4395gh3vnb-0w8rgbf0-wbei-0ebf-0h.obs.ap-southeast-3.myhuaweicloud.com/bv3oi84-3gh95-08fihrbwfn-0w8irf-0wbe-fhw.html?awsaccesskeyid=sklopwpnkxzh1ld7wjsq&expires=1677693542&signature=cqseh6id8w3eh5u2o6sj5y73nga%3d
🔍 處理中 [8953/100000]: https://www.corvettemuseum.org
🔍 處理中 [8954/100000]: https://www.handimania.com
🔍 處理中 [8955/100000]: https://www.drumcorpsworld.com
🔍 處理中 [8956/100000]: http://www.saiconsard.co.jp.tbcowj.top/ai/sign.php
🔍 處理中 [8957/100000]: http://bt-login-page-103284.weeblysite.com/
🔍 處理中 [8958/100000]: https://www.carbontrust.com
🔍 處理中 [8959/1000

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=Error("TypeError: Cannot read properties of null (reading 'scrollHeight')\n    at eval (eval at evaluate (:290:30), <anonymous>:1:34)\n    at eval (<anonymous>)\n    at UtilityScript.evaluate (<anonymous>:290:30)\n    at UtilityScript.<anonymous> (<anonymous>:1:44)")>
playwright._impl._errors.Error: TypeError: Cannot read properties of null (reading 'scrollHeight')
    at eval (eval at evaluate (:290:30), <anonymous>:1:34)
    at eval (<anonymous>)
    at UtilityScript.evaluate (<anonymous>:290:30)
    at UtilityScript.<anonymous> (<anonymous>:1:44)


⏰ [上帝大限] 第 9042 筆 URL (https://www.visitgilgitbaltistan.gov.pk) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9043 筆 URL (https://www.themandagies.com) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9044 筆 URL (http://www.543t.com) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9046 筆 URL (https://www.beardybird.com) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9047 筆 URL (https://www.kuglermaag.com) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9048 筆 URL (https://www.australia108.com.au) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9067 筆 URL (https://www.flykickdesign.com) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9050 筆 URL (https://www.cinerex.fr) 卡死超過 90 秒，已強制拔管！
⏰ [上帝大限] 第 9052 筆 URL (https://www.karmapa.org.nz) 卡死超過 90 秒，已強制拔管！
🔍 處理中 [9079/100000]: https://www.estatebeads.com
🔍 處理中 [9080/100000]: https://www.bakerperkinsgroup.com
🔍 處理中 [9081/100000]: http://www.customfencer.com/mygov.ato/mygov/otp2.html
🔍 處理中 [9082/100000]: https://www.britishdragons.org
🔍 處理中 [9083/100000]: http://www.intersys32.com
🔍 處理中 [9084/100000]: https://www.takamaka.io
🔍 處理中 [9085/100000]: https://www.honors.ufl.edu
🔍 處理中